# Data Ingestion, Cleaning & Preprocessing with Pandas

This notebook cleans a deliberately messy 12,620-row retail transactions dataset. It demonstrates ingestion, data-quality auditing, missing-value treatment, duplicate removal, type correction, outlier handling, feature engineering, and CSV export.

**Expected deliverable:** `clean_dataset.csv`

In [ ]:
# Data Ingestion, Cleaning & Preprocessing with Pandas

import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE = "raw_retail_sales.csv"
CLEAN_FILE = "clean_dataset.csv"

# 1. Load raw dataset
raw = pd.read_csv(RAW_FILE)
print("Raw shape:", raw.shape)
display(raw.head())

# 2. Initial data-quality audit
print("\nMissing values:")
display(raw.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nDuplicate rows:", raw.duplicated().sum())
print("\nData types:")
display(raw.dtypes.astype(str).to_frame("dtype"))

# 3. Standardize column names and text fields
df = raw.copy()
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_"))

for c in ["customer_id","region","category","product","payment_method","sales_channel","returned"]:
    df[c] = df[c].astype("string").str.strip()

df["region"] = df["region"].str.title()
df["payment_method"] = df["payment_method"].str.title()

# 4. Correct data types
df["transaction_date"] = pd.to_datetime(df["transaction_date"], dayfirst=True, errors="coerce")

for c in ["unit_price","sales","cost","profit"]:
    df[c] = (df[c].astype("string")
             .str.replace(r"[$,]", "", regex=True)
             .str.strip())
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["discount"] = (df["discount"].astype("string")
                  .str.replace("%", "", regex=False).str.strip())
df["discount"] = pd.to_numeric(df["discount"], errors="coerce")
df.loc[df["discount"] > 1, "discount"] /= 100

# 5. Remove duplicate records
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().copy()
print("Removed duplicates:", duplicate_count)

# 6. Handle invalid values and outliers
df.loc[df["quantity"] <= 0, "quantity"] = np.nan
q_low, q_high = df["quantity"].quantile([0.01, 0.99])
df["quantity"] = df["quantity"].clip(q_low, q_high).round().astype("Int64")

# 7. Impute missing values
for c in ["region","category","payment_method"]:
    df[c] = df[c].fillna(df[c].mode()[0])
df["customer_id"] = df["customer_id"].fillna("UNKNOWN")
df["transaction_date"] = df["transaction_date"].fillna(df["transaction_date"].median())
for c in ["unit_price","discount"]:
    df[c] = df[c].fillna(df[c].median())

# Recalculate financial measures so they are internally consistent
df["sales"] = (df["quantity"].astype(float) * df["unit_price"] * (1 - df["discount"])).round(2)
df["cost"] = df["cost"].fillna(df["sales"] * 0.70).round(2)
df["profit"] = (df["sales"] - df["cost"]).round(2)

# 8. Feature engineering
df["year"] = df["transaction_date"].dt.year.astype("int64")
df["month"] = df["transaction_date"].dt.month.astype("int64")
df["year_month"] = df["transaction_date"].dt.to_period("M").astype(str)
df["profit_margin"] = np.where(df["sales"] != 0, df["profit"] / df["sales"], 0).round(4)

# 9. Final standardized output
df["transaction_date"] = df["transaction_date"].dt.strftime("%Y-%m-%d")
ordered = ['transaction_id', 'transaction_date', 'year', 'month', 'year_month', 'customer_id', 'region', 'category', 'product', 'quantity', 'unit_price', 'discount', 'sales', 'cost', 'profit', 'profit_margin', 'payment_method', 'sales_channel', 'returned']
df = df[ordered].sort_values("transaction_id").reset_index(drop=True)
df.to_csv(CLEAN_FILE, index=False)

# 10. Before vs after proof
print("Before cleaning:", raw.shape)
print("After cleaning :", df.shape)
print("Missing cells after cleaning:", df.isna().sum().sum())
print("Duplicate rows after cleaning:", df.duplicated().sum())
display(df.head())
display(df.describe(include="all").T.head(20))


## Cleaning summary

- Raw dataset: **12,620 rows × 15 columns**
- Exact duplicate rows removed: **120**
- Clean dataset: **12,500 rows × 19 columns**
- Missing cells after cleaning: **0**
- Engineered fields: `year`, `month`, `year_month`, and `profit_margin`
- Output file: `clean_dataset.csv`

In [ ]:
# Optional verification when the clean CSV is already present
clean = pd.read_csv(CLEAN_FILE)
print('Rows:', len(clean))
print('Columns:', len(clean.columns))
print('Missing cells:', clean.isna().sum().sum())
print('Duplicate rows:', clean.duplicated().sum())
display(clean.head())